# 01. 문서 파싱 — PDF · Excel

임베딩 파이프라인의 1단계. 문서에서 텍스트와 구조를 추출해 **중간 산출물(jsonl)** 로 저장한다.

파싱과 임베딩을 분리하는 이유: 임베딩 모델을 바꿔도 파싱을 반복하지 않기 위해 (노트 6절 참고).

1. 실습용 샘플 문서 생성 (한국어 PDF + Excel)
2. PDF 파싱 — PyMuPDF
3. Excel 파싱 — 행 단위 직렬화
4. 중간 산출물 저장

## 1. 샘플 문서 생성

실제 문서가 있으면 이 단계를 건너뛰고 `data/` 에 파일을 넣으면 된다.

> fpdf2로 한국어 PDF를 만들려면 한글 TTF 폰트가 필요하다 — Windows 기본 폰트(맑은 고딕)를 사용.

In [1]:
from pathlib import Path
from fpdf import FPDF
from fpdf.enums import XPos, YPos

DATA = Path("../data")
DATA.mkdir(exist_ok=True)

sections = {
    "1. 개요": "본 매뉴얼은 컨베이어 제어 시스템 CX-200의 설치와 운영 절차를 설명한다. "
               "CX-200은 PLC 기반 제어기로, Mitsubishi 3E Frame 프로토콜로 통신한다.",
    "2. 설치": "장비는 반드시 전원 차단 상태에서 설치한다. 제어반은 분전반에서 3m 이내에 배치하고, "
               "통신 케이블은 동력선과 30cm 이상 이격한다. 접지 저항은 100옴 이하를 유지해야 한다.",
    "3. 운영": "기동 전 비상정지 버튼의 동작을 확인한다. 운전 모드는 수동/자동 두 가지이며, "
               "자동 모드에서는 상위 MES 시스템의 지시에 따라 동작한다. 이상 발생 시 알람 코드가 HMI에 표시된다.",
    "4. 유지보수": "벨트 장력은 월 1회 점검한다. 모터 베어링은 6개월마다 그리스를 주입한다. "
                  "알람 코드 E-04는 인버터 과열이며, 냉각팬 필터 청소 후에도 재발하면 인버터를 교체한다.",
}

pdf = FPDF()
pdf.add_font("Malgun", fname=r"C:\Windows\Fonts\malgun.ttf")
pdf.add_page()
# multi_cell 은 출력 후 커서를 셀 오른쪽 끝(XPos.RIGHT)으로 옮기는 게 기본값이라
# w=0 으로 연속 호출하면 가용 폭이 0이 되어 예외가 난다 → 왼쪽 여백·다음 줄 이동을 명시
for title, body in sections.items():
    pdf.set_font("Malgun", size=14)
    pdf.multi_cell(0, 10, title, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.set_font("Malgun", size=11)
    pdf.multi_cell(0, 8, body, new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    pdf.ln(4)

pdf_path = DATA / "cx200_manual.pdf"
pdf.output(str(pdf_path))
print("created:", pdf_path)

MERG NOT subset; don't know how to subset; dropped


created: ..\data\cx200_manual.pdf


In [2]:
import pandas as pd

df = pd.DataFrame(
    {
        "부품명": ["컨베이어 벨트 B-1200", "인버터 팬 필터", "모터 베어링 6204", "비상정지 스위치", "근접센서 PR-08"],
        "부품코드": ["BLT-1200", "FLT-INV", "BRG-6204", "SW-ESTOP", "SNS-PR08"],
        "재고수량": [3, 12, 8, 5, 20],
        "단가(원)": [450000, 8000, 15000, 32000, 27000],
        "교체주기": ["24개월", "3개월", "6개월", "점검시", "고장시"],
    }
)

xlsx_path = DATA / "spare_parts.xlsx"
df.to_excel(xlsx_path, sheet_name="예비부품", index=False)
df

,부품명,부품코드,재고수량,단가(원),교체주기
0,컨베이어 벨트 B-1200,BLT-1200,3,450000,24개월
1,인버터 팬 필터,FLT-INV,12,8000,3개월
2,모터 베어링 6204,BRG-6204,8,15000,6개월
3,비상정지 스위치,SW-ESTOP,5,32000,점검시
4,근접센서 PR-08,SNS-PR08,20,27000,고장시


## 2. PDF 파싱 (PyMuPDF)

`page.get_text()` 가 빈 문자열이면 스캔 PDF(이미지)라서 OCR이 필요하다는 신호다.

In [3]:
import pymupdf

doc = pymupdf.open(pdf_path)
pdf_text = "\n".join(page.get_text() for page in doc)
doc.close()

assert pdf_text.strip(), "텍스트가 비어 있음 → 스캔 PDF, OCR 필요"
print(pdf_text[:500])

1. 개요
본 매뉴얼은 컨베이어 제어 시스템 CX-200의 설치와 운영 절차를 설명한다. CX-200은 PLC 기반 제어기로,
Mitsubishi 3E Frame 프로토콜로 통신한다.
2. 설치
장비는 반드시 전원 차단 상태에서 설치한다.  제어반은 분전반에서 3m  이내에 배치하고,  통신 케이블은
동력선과 30cm 이상 이격한다. 접지 저항은 100옴 이하를 유지해야 한다.
3. 운영
기동 전 비상정지 버튼의 동작을 확인한다. 운전 모드는 수동/자동 두 가지이며, 자동 모드에서는 상위 MES
시스템의 지시에 따라 동작한다. 이상 발생 시 알람 코드가 HMI에 표시된다.
4. 유지보수
벨트 장력은 월 1회 점검한다.  모터 베어링은 6개월마다 그리스를 주입한다.  알람 코드 E-04는 인버터
과열이며, 냉각팬 필터 청소 후에도 재발하면 인버터를 교체한다.



## 3. Excel 파싱 — 행 단위 직렬화

시트를 통째로 텍스트화하면 헤더와 값의 연결이 끊긴다.
각 행을 `"헤더: 값 | 헤더: 값"` 문장으로 변환해 **행 하나 = 검색 문서 하나**로 만든다.

In [4]:
df = pd.read_excel(xlsx_path, sheet_name="예비부품")

excel_rows = []
for i, row in df.iterrows():
    text = " | ".join(f"{col}: {row[col]}" for col in df.columns)
    excel_rows.append({"text": text, "sheet": "예비부품", "row": int(i) + 2})  # +2: 헤더 다음, 1-base

for r in excel_rows[:3]:
    print(r["text"])

부품명: 컨베이어 벨트 B-1200 | 부품코드: BLT-1200 | 재고수량: 3 | 단가(원): 450000 | 교체주기: 24개월
부품명: 인버터 팬 필터 | 부품코드: FLT-INV | 재고수량: 12 | 단가(원): 8000 | 교체주기: 3개월
부품명: 모터 베어링 6204 | 부품코드: BRG-6204 | 재고수량: 8 | 단가(원): 15000 | 교체주기: 6개월


## 4. 중간 산출물 저장 (jsonl)

문서 종류가 달라도 이후 단계(청킹·임베딩)가 동일하게 처리할 수 있도록 **공통 스키마**로 저장한다:
`{source, type, text, metadata}`

In [5]:
import json

records = [
    {
        "source": pdf_path.name,
        "type": "pdf",
        "text": pdf_text,
        "metadata": {"title": "CX-200 매뉴얼"},
    }
]
records += [
    {
        "source": xlsx_path.name,
        "type": "excel-row",
        "text": r["text"],
        "metadata": {"title": "예비부품 목록", "sheet": r["sheet"], "row": r["row"]},
    }
    for r in excel_rows
]

out = DATA / "parsed.jsonl"
with open(out, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"saved {len(records)} records → {out}")

saved 6 records → ..\data\parsed.jsonl
